# Importing libraries


In [ ]:
!pip install opencv-python deepface dlib mtcnn flask

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.0 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=918f2e41183bd2838984131fce015b5122bcab45b241e3622daea6d0c10980ce
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire


#Extracting Face From Adhaar Card

In [ ]:
import cv2
from deepface import DeepFace

def extract_face(image_path):
    img = cv2.imread(image_path)
    detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = detector.detectMultiScale(gray, 1.1, 4)

    for (x, y, w, h) in faces:
        face = img[y:y+h, x:x+w]
        return face
    return None


25-04-02 14:15:58 - Directory /root/.deepface has been created
25-04-02 14:15:58 - Directory /root/.deepface/weights has been created


# Capturing Real Time Image

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
import PIL.Image

# JavaScript code to access webcam
def take_photo(filename='live_image.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const video = document.createElement('video');
            const capture = document.createElement('button');
            capture.textContent = 'Capture';
            div.appendChild(video);
            div.appendChild(capture);
            document.body.appendChild(div);

            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();

            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getTracks().forEach(track => track.stop());
            div.remove();

            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')

    display(js)
    data = eval_js('takePhoto({})'.format(quality))

    # Decode base64 image
    img_data = data.split(',')[1]
    binary = b64decode(img_data)

    with open(filename, 'wb') as f:
        f.write(binary)

    return filename


In [ ]:
photo_path = take_photo()
print(f"Saved live image as {photo_path}")


<IPython.core.display.Javascript object>

Saved live image as live_image.jpg


# Compare Faces

In [ ]:
def verify_face(aadhaar_face, live_face):
    result = DeepFace.verify(aadhaar_face, live_face, model_name='Facenet')
    return result["verified"]


In [ ]:
import time
from deepface import DeepFace

def verify_face(aadhaar_face, live_face, threshold=0.4, test_samples=10):
    start_time = time.time()

    total_tests = test_samples
    correct_verifications = 0
    false_accepts = 0
    false_rejects = 0

    for _ in range(total_tests):
        result = DeepFace.verify(aadhaar_face, live_face, model_name='Facenet')

        similarity = result["distance"]
        verified = result["verified"]

        if similarity < threshold and verified:
            correct_verifications += 1
        elif similarity > threshold and verified:
            false_accepts += 1
        elif similarity < threshold and not verified:
            false_rejects += 1


    accuracy = (correct_verifications / total_tests) * 100
    far = (false_accepts / total_tests) * 100
    frr = (false_rejects / total_tests) * 100
    processing_time = round(time.time() - start_time, 3)


    print("🔍 **Face Verification Result Analysis:**")
    print(f" Verified: {verified}")
    print(f" Similarity Distance: {similarity:.4f}")
    print(f" Accuracy: {accuracy:.2f}%")
    print(f" False Rejection Rate (FRR): {frr:.2f}%")
    print(f" False Acceptance Rate (FAR): {far:.2f}%")
    print(f" Processing Time: {processing_time} seconds")

    return verified, accuracy, far, frr, processing_time


aadhaar_face = "/content/aadhaar_face.jpg"
live_face = "live_image.jpg"

verify_face(aadhaar_face, live_face)


🔍 **Face Verification Result Analysis:**
✅ Verified: True
🔢 Similarity Distance: 0.2298
🎯 Accuracy: 100.00%
📉 False Rejection Rate (FRR): 0.00%
📈 False Acceptance Rate (FAR): 0.00%
⏳ Processing Time: 13.271 seconds


(True, 100.0, 0.0, 0.0, 13.271)

# Main Function

In [ ]:
from deepface import DeepFace

aadhaar_image = "/content/Screenshot.png"
live_image = "/content/live_image.jpg"


result = DeepFace.verify(aadhaar_image, live_image, model_name="Facenet")

if result["verified"]:
    print("Face Verification Successful")
else:
    print(" e-KYC Failed")


 e-KYC Failed


#To see Scanned Image

In [ ]:
import cv2
from deepface import DeepFace

def extract_face(image_path, output_path="extracted_face.jpg"):
    img = cv2.imread(image_path)

    if img is None:
        print(" Error: Image not found or unable to read!")
        return None

    detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4)

    if len(faces) == 0:
        print(" No face detected!")
        return None

    for (x, y, w, h) in faces:
        face = img[y:y+h, x:x+w]

        cv2.imwrite(output_path, face)
        print(f" Face saved as {output_path}")
        return output_path

    return None

aadhaar_image = "/content/Screenshot.png"
saved_face_path = extract_face(aadhaar_image, "aadhaar_face.jpg")

if saved_face_path:
    print(f"Extracted face saved at: {saved_face_path}")


 Face saved as aadhaar_face.jpg
Extracted face saved at: aadhaar_face.jpg
